# 00F · 数据闭环、Safety、评测与部署：模型开发不是训练结束

自动驾驶岗位中的“模型开发”通常不是只改网络和 loss。一次线上/仿真失败需要经过：场景定位 → 数据切片 → 重放 → root cause → 修复模型或规则 → 重新评测 → runtime 回归 → 发布门禁。

本节先建立四种证据的区别：

- **open-loop**：模型对固定记录的预测/检测是否正确；
- **closed-loop**：策略动作会不会改变后续场景和风险；
- **safety evidence**：故障、退化、TTC、fallback、ODD exit 是否可检测且有响应；
- **deployment evidence**：batch=1 latency、p95/p99、显存、吞吐、量化误差和版本回归。

这些证据共同决定“模型能否进入下一轮验证”，而不是某一个漂亮的平均分。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(31)
scenarios = pd.DataFrame([
    {"scenario_id": "sunny_dense", "weather": "sunny", "density": "dense", "sensor_fault": "none"},
    {"scenario_id": "rain_sparse", "weather": "rain", "density": "sparse", "sensor_fault": "none"},
    {"scenario_id": "night_occlusion", "weather": "night", "density": "dense", "sensor_fault": "camera"},
    {"scenario_id": "gnss_outage", "weather": "sunny", "density": "medium", "sensor_fault": "gnss"},
    {"scenario_id": "cut_in", "weather": "sunny", "density": "medium", "sensor_fault": "none"},
])
scenarios["episodes"] = [80, 70, 45, 35, 50]
scenarios["collision_rate"] = [0.01, 0.04, 0.15, 0.08, 0.12]
scenarios["p95_latency_ms"] = [82, 90, 108, 99, 87]
display(scenarios)
print("aggregate collision rate:", np.average(scenarios.collision_rate, weights=scenarios.episodes).round(3))
print("worst slice:", scenarios.loc[scenarios.collision_rate.idxmax(), "scenario_id"])


In [ ]:
def slice_report(min_collision_rate=0.08, max_latency_ms=100):
    risky = scenarios[(scenarios.collision_rate >= min_collision_rate) | (scenarios.p95_latency_ms >= max_latency_ms)]
    print(risky[["scenario_id", "weather", "sensor_fault", "collision_rate", "p95_latency_ms"]].to_string(index=False))
    print("data-loop action: replay → label/root-cause → targeted training or rule → regression gate")

slice_report()


In [ ]:
from ipywidgets import FloatSlider, IntSlider, interact

def gate_experiment(collision_threshold=0.08, latency_budget_ms=100, fallback_recall=0.92):
    collision_pass = scenarios.collision_rate < collision_threshold
    latency_pass = scenarios.p95_latency_ms < latency_budget_ms
    safety_pass = fallback_recall >= 0.95
    report = scenarios[["scenario_id"]].copy()
    report["collision_gate"] = collision_pass
    report["latency_gate"] = latency_pass
    report["safety_gate"] = safety_pass
    display(report)
    print("release decision:", "candidate for next validation stage" if report.iloc[:, 1:].all().all() else "hold / investigate slices")

interact(
    gate_experiment,
    collision_threshold=FloatSlider(min=0.02, max=0.2, step=0.01, value=0.08, description="collision"),
    latency_budget_ms=IntSlider(min=70, max=140, step=5, value=100, description="p95 / ms"),
    fallback_recall=FloatSlider(min=0.8, max=1.0, step=0.01, value=0.92, description="fallback recall"),
)


## 领域检查点

- 为什么平均 collision rate 可能掩盖 night/occlusion 或 GNSS outage 的风险？
- 一个模型 accuracy 提升但 p99 latency 超预算，是否应该发布？需要谁来决定？
- corner-case mining 产出的 hard slice 如何回到数据、训练、场景回放和 regression gate？

**下一步**：`10–12` 进入数据管线、closed-loop metrics 和 corner-case mining；`15` 进入 runtime；`18–19` 进入安全状态机和 capstone。
